# 🧪 RecallTrace — LLM Agent Training
**Unsloth + TRL SFT on Expert Demonstrations**

> ⚠️ **Requires GPU**: Runtime → Change runtime type → **T4 GPU** → Save

This notebook:
1. Installs all dependencies
2. Clones the RecallTrace repo from HF Space
3. Generates expert demonstrations via the heuristic policy
4. Fine-tunes `Qwen2.5-0.5B-Instruct` with Unsloth + TRL
5. Evaluates baseline vs trained agent
6. Pushes the trained model to `ms-shamanth/recalltrace-investigator`

## Step 0 — Verify GPU

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU found! Go to Runtime → Change runtime type → T4 GPU → Save, then reconnect.")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install unsloth "trl>=0.12" datasets accelerate openai
print("✅ Dependencies installed")

## Step 2 — Clone RecallTrace from HF Space

In [ ]:
import os
if not os.path.exists("recalltrace-openenv"):
    !git clone https://huggingface.co/spaces/ms-shamanth/recalltrace-openenv
    print("✅ Repo cloned")
else:
    !git -C recalltrace-openenv pull
    print("✅ Repo updated")
%cd recalltrace-openenv

## Step 3 — Set HF Token (for pushing trained model)

In [ ]:
# Option A: paste token directly (delete after use)
HF_TOKEN = "hf_..."  # <-- replace with your token from https://huggingface.co/settings/tokens

# Option B: use Colab secrets (recommended)
# from google.colab import userdata
# HF_TOKEN = userdata.get('HF_TOKEN')

os.environ["HF_TOKEN"] = HF_TOKEN
print("✅ HF_TOKEN set")

## Step 4 — Run Training

In [ ]:
!python train_trl.py \
    --episodes 300 \
    --epochs 3 \
    --eval-episodes 30 \
    --push-model \
    --hub-model-id ms-shamanth/recalltrace-investigator

## Step 5 — View Training Plots

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for plot in sorted(Path("plots").glob("trl_*.png")):
    print(f"\n📊 {plot.name}")
    display(Image(str(plot)))